In [0]:
from pyspark.sql.functions import (
    col, trim, upper, when, coalesce, lit,
    to_date, abs as spark_abs, current_timestamp
)
from pyspark.sql.types import DoubleType, IntegerType

print("Starting Silver transformation...\n")



In [0]:

# 1. CUSTOMERS 

print("Processing Customers...")
df_customers_raw = spark.table("globaldistributor.01_bronze.customers")

df_customers_silver = df_customers_raw \
    .dropDuplicates(["customer_id"]) \
    .filter(col("customer_id").isNotNull()) \
    .withColumn("customer_name", trim(col("customer_name"))) \
    .withColumn("customer_type", upper(trim(col("customer_type")))) \
    .withColumn("country", upper(trim(col("country")))) \
    .withColumn("signup_date", to_date(col("signup_date"))) \
    .withColumn("silver_updated_at", current_timestamp())

df_customers_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("globaldistributor.02_silver.customers")

print(f"Silver customers: {df_customers_silver.count()} rows\n")



In [0]:

# 2. PRODUCTS 

print("Processing Products...")
df_products_raw = spark.table("globaldistributor.01_bronze.products")

df_products_silver = df_products_raw \
    .dropDuplicates(["product_id"]) \
    .filter(col("product_id").isNotNull()) \
    .withColumn("product_name", trim(col("product_name"))) \
    .withColumn("category", upper(trim(col("category")))) \
    .withColumn("cost_price", col("cost_price").cast(DoubleType())) \
    .filter(col("cost_price") >= 0) \
    .withColumn("silver_updated_at", current_timestamp())

df_products_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("globaldistributor.02_silver.products")

print(f"Silver products: {df_products_silver.count()} rows\n")

In [0]:

# 3. REGIONS 

print("Processing Regions...")
df_regions_raw = spark.table("globaldistributor.01_bronze.regions")

df_regions_silver = df_regions_raw \
    .dropDuplicates(["region_code"]) \
    .filter(col("region_code").isNotNull()) \
    .withColumn("region_name", trim(col("region_name"))) \
    .withColumn("country", upper(trim(col("country")))) \
    .withColumn("silver_updated_at", current_timestamp())

df_regions_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("globaldistributor.02_silver.regions")

print(f"Silver regions: {df_regions_silver.count()} rows\n")

In [0]:

# 4. INVOICES 

print("Processing Invoices...")
df_invoices_raw = spark.table("globaldistributor.01_bronze.invoices")

# Standardize invoice status values
valid_statuses = ["PAID", "PENDING", "CANCELLED", "OVERDUE"]

df_invoices_silver = df_invoices_raw \
    .dropDuplicates(["invoice_id"]) \
    .filter(col("invoice_id").isNotNull()) \
    .filter(col("customer_id").isNotNull()) \
    .withColumn("invoice_status", upper(trim(col("invoice_status")))) \
    .withColumn("invoice_status",
        when(col("invoice_status").isin(valid_statuses), col("invoice_status"))
        .otherwise(lit("UNKNOWN"))
    ) \
    .withColumn("invoice_date", to_date(col("invoice_date"), "yyyy/M/d")) \
    .withColumn("currency", upper(trim(col("currency")))) \
    .withColumn("region", upper(trim(col("region")))) \
    .withColumn("silver_updated_at", current_timestamp())

df_invoices_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("globaldistributor.02_silver.invoices")

print(f"Silver invoices: {df_invoices_silver.count()} rows\n")

In [0]:
# 5. INVOICE LINE ITEMS (with USD Conversion)

print("Processing Invoice Line Items...")
df_line_raw = spark.table("globaldistributor.01_bronze.invoice_line_items")

# Clean line items
df_line_silver = df_line_raw \
    .dropDuplicates(["invoice_line_id"]) \
    .filter(col("invoice_line_id").isNotNull()) \
    .filter(col("invoice_id").isNotNull()) \
    .filter(col("product_id").isNotNull()) \
    .withColumn("quantity", spark_abs(col("quantity").cast(IntegerType()))) \
    .filter(col("quantity") > 0) \
    .withColumn("unit_price", col("unit_price").cast(DoubleType())) \
    .filter(col("unit_price") >= 0) \
    .withColumn("discount", coalesce(col("discount").cast(DoubleType()), lit(0.0))) \
    .withColumn("line_total",
        col("quantity") * col("unit_price") - col("discount")
    )

# Join with invoices to get currency and date
df_invoices = spark.table("globaldistributor.02_silver.invoices") \
    .select("invoice_id", "currency", "invoice_date")

df_line_with_currency = df_line_silver.join(
    df_invoices,
    "invoice_id",
    "left"
)

# Join with exchange rates to get USD conversion rate
df_fx = spark.table("globaldistributor.02_silver.exchange_rates") \
    .select(
        col("currency").alias("fx_currency"),
        col("date").alias("fx_date"),
        col("rate_to_usd")
    )

df_line_with_usd = df_line_with_currency.join(
    df_fx,
    (col("currency") == col("fx_currency")) & (col("invoice_date") == col("fx_date")),
    "left"
)

# Calculate USD amount (if no rate found, keep original amount as fallback)
df_line_final = df_line_with_usd \
    .withColumn("line_total_usd",
        when(col("rate_to_usd").isNotNull(), col("line_total") * col("rate_to_usd"))
        .otherwise(col("line_total"))  # Fallback to original if no rate
    ) \
    .withColumn("silver_updated_at", current_timestamp()) \
    .select(
        "invoice_line_id",
        "invoice_id",
        "product_id",
        "quantity",
        "unit_price",
        "discount",
        "line_total",
        "currency",
        "line_total_usd",
        "silver_updated_at"
    )

df_line_final.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("globaldistributor.02_silver.invoice_line_items")

print(f"Silver line items: {df_line_final.count()} rows")
print(f"Currency conversion applied using exchange rates\n")

In [0]:

# 6. PAYMENTS 
print("Processing Payments...")
df_payments_raw = spark.table("globaldistributor.01_bronze.payments")

df_payments_silver = df_payments_raw \
    .dropDuplicates(["payment_id"]) \
    .filter(col("payment_id").isNotNull()) \
    .filter(col("invoice_id").isNotNull()) \
    .withColumn("payment_amount", col("payment_amount").cast(DoubleType())) \
    .filter(col("payment_amount") > 0) \
    .withColumn("payment_date", to_date(col("payment_date"), "yyyy/M/d")) \
    .withColumn("payment_method", upper(trim(col("payment_method")))) \
    .withColumn("silver_updated_at", current_timestamp())

df_payments_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("globaldistributor.02_silver.payments")

print(f"Silver payments: {df_payments_silver.count()} rows\n")

In [0]:
# 7. EXCHANGE RATES 

print("Processing Exchange Rates...")
df_fx_raw = spark.table("globaldistributor.01_bronze.exchange_rates")

df_fx_silver = df_fx_raw \
    .dropDuplicates(["currency", "date"]) \
    .withColumn("currency", upper(trim(col("currency")))) \
    .withColumn("date", to_date(col("date"), "yyyy/M/d")) \
    .withColumn("rate_to_usd", col("rate_to_usd").cast(DoubleType())) \
    .filter(col("rate_to_usd") > 0) \
    .withColumn("silver_updated_at", current_timestamp())

df_fx_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("globaldistributor.02_silver.exchange_rates")

print(f" Silver exchange rates: {df_fx_silver.count()} rows\n")

print("="*60)
print("All Silver tables successfully created")
print("="*60)

In [0]:
spark.sql("SHOW TABLES IN globaldistributor.02_silver").display()


In [0]:
# Verify Currency Conversion
print("\n" + "="*60)
print("CURRENCY CONVERSION VERIFICATION")
print("="*60)

df_verify = spark.table("globaldistributor.02_silver.invoice_line_items")

# Show conversion by currency
print("\nRevenue by Currency (Original vs USD):")
df_verify.groupBy("currency") \
    .agg(
        {"line_total": "sum", "line_total_usd": "sum"}
    ) \
    .withColumnRenamed("sum(line_total)", "total_original_currency") \
    .withColumnRenamed("sum(line_total_usd)", "total_usd") \
    .orderBy("currency") \
    .display()

print(f"\nTotal Revenue (USD): ${df_verify.agg({'line_total_usd': 'sum'}).collect()[0][0]:,.2f}")
print("All line items now have standardized USD values for KPI calculations\n")